# Export to Hugging Face Hub (SFT + DPO model)

Uploads the final model, **SFT + DPO merged** (`outputs/dpo_qwen2.5-0.5b/merged/`), to Hugging Face, so anyone can load it with **one line**:
`AutoModelForCausalLM.from_pretrained("your-name/your-model")`.

**Steps:** 1. config → 2. check the files → 3. set greedy decoding defaults → 4. test it twice (a normal call + a "no tool fits" case) → 5. write a model card with the base → SFT → DPO results → 6. log in → 7. upload → 8. load it back.

The model card reads every number from the eval files (`10_eval_base_sft_dpo` for DPO, `03`/`04` for base and SFT), so **run notebook 10 first**.
(The SFT-only export is the original `05_export_huggingface.ipynb`.)

You need a free Hugging Face account and an access token with **write** permission (huggingface.co → Settings → Access Tokens).


## 1. Config
Set `HF_REPO_ID` to `your-hf-username/model-name`. A new repo name (here `...-instructv2-fc`) keeps the SFT model page as it is. Keep `PRIVATE = True` until you've checked the upload; you can make it public later on the website.

In [ ]:
import json, sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
NOTEBOOKS = REPO_ROOT / "notebooks"

RUN_NAME = "dpo_qwen2.5-0.5b"            # final model: SFT + DPO (merged)
MERGED_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "merged"
RESULTS_ROOT = NOTEBOOKS / "data" / "eval" / "results"
RESULTS_DIR = RESULTS_ROOT / RUN_NAME          # metrics.json written by notebook 10

HF_REPO_ID = "SwapniL-pr1117/qwen2.5-0.5b-instructv2-fc"   # <-- change this
PRIVATE = True

print("Model folder:", MERGED_DIR, "| exists:", MERGED_DIR.exists())
assert (RESULTS_DIR / "metrics.json").exists(), "run 10_eval_base_sft_dpo first: the model card reads its metrics.json"

## 2. Check what we're uploading
The folder must contain the weights (`model.safetensors`), the architecture (`config.json`) and the tokenizer. The safetensors header tells us the number type and the parameter count without loading the model.

In [ ]:
import math
import struct

for f in sorted(MERGED_DIR.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1024**2:8.1f} MB")

with open(MERGED_DIR / "model.safetensors", "rb") as fh:
    header = json.loads(fh.read(struct.unpack("<Q", fh.read(8))[0]))
header.pop("__metadata__", None)
n_params = sum(math.prod(t["shape"]) for t in header.values())
dtypes = {t["dtype"] for t in header.values()}
print(f"\n{len(header)} tensors | {n_params / 1e6:.0f}M parameters | dtype: {dtypes}")

## 3. Set greedy decoding as the default
`merged/generation_config.json` still holds Qwen's **chat** defaults (`temperature 0.7`, `top_p 0.8`, `repetition_penalty 1.1`, sampling on).
Anyone calling plain `model.generate()` would get those, i.e. random and lower accuracy than our greedy eval.
`repetition_penalty` is especially bad for JSON, which repeats `"`, `{` and `:` all the time.
We overwrite it with greedy decoding so the model behaves like it did in evaluation **out of the box**.


In [ ]:
gen_path = MERGED_DIR / "generation_config.json"
gen = json.loads(gen_path.read_text())
print("before:", {k: gen.get(k) for k in ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty")})

for k in ("temperature", "top_p", "top_k"):      # only meaningful when sampling
    gen.pop(k, None)
gen["do_sample"] = False
gen["repetition_penalty"] = 1.0

gen_path.write_text(json.dumps(gen, indent=2))
print("after: ", {k: gen.get(k) for k in ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty")})


## 4. Test it before uploading
Load with plain `transformers` and ask two questions in the **trained prompt format** (`build_messages`: system prompt + tools + request):
- a normal request → we expect a JSON tool call,
- *"Tell me a joke"* with only a weather tool → we expect `[]` (what DPO taught it; SFT called `get_weather` here).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.prompts import build_messages
from src.evaluate import extract_json

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype="auto", device_map="auto")

weather_tool = [{"name": "get_weather", "description": "Current weather for a city.",
                 "parameters": {"city": {"type": "str", "description": "City name"}}}]
record = {"query": "What's the weather like in Pune right now?", "tools": weather_tool}     # expect a call
no_fit = {"query": "Tell me a joke", "tools": weather_tool}                                 # expect []

for name, rec in [("normal", record), ("no tool fits", no_fit)]:
    inputs = tokenizer.apply_chat_template(build_messages(rec), add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(model.device)
    out = getattr(model, "generate")(**inputs, max_new_tokens=128, do_sample=False)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"[{name}] {rec['query']}\n  MODEL : {answer}\n  PARSED: {extract_json(answer)}")

del model
torch.cuda.empty_cache()

## 5. Write the model card (`README.md`)
The model card is the page people see on Hugging Face. The block at the top (between `---`) is metadata: base model, license and tags make it searchable.
- **Results** compares base → SFT → DPO on the same frozen test sets and tells the story in two lines: SFT learned to call tools but forgot how to say "no tool fits"; DPO fixed that *and* improved accuracy.
- The **prompt format** section matters most: without it, nobody can use the model correctly.
- **Training** explains how the DPO pairs were built (the model's own mistakes), the interesting part for readers.
- All numbers are read from the `metrics.json` files, never typed by hand, and the **limitations** stay honest (abstention ~82%, not 100%; rare repetition loops; mostly familiar tools in the test set).

In [ ]:
from src.prompts import SYSTEM_PROMPT

GITHUB_URL = "https://github.com/SwapnilGEU/Toolcaller-SLM"   # <-- your repo

# all numbers come from the eval files (notebooks 03, 04, 10), never typed by hand
m = json.loads((RESULTS_DIR / "metrics.json").read_text())                 # DPO (notebook 10)
m_sft = json.loads((RESULTS_ROOT / "sft_qwen2.5-0.5b" / "metrics.json").read_text())
m_base = json.loads((RESULTS_ROOT / "base_qwen2.5-0.5b" / "metrics.json").read_text())
S = {"base": m_base["summary"], "sft": m_sft["summary"], "dpo": m["summary"]}
ab, oa, chat = m["abstain_v1"], m["over_abstention_pct"], m["chat_v2"]["tool_json_pct"]
vs_sft, vs_base, bd = m["vs_sft"], m["vs_base"], m["breakdowns"]

def metric_row(name, key):
    return f"| {name} | " + " | ".join(f"{S[k][key]['value']:.1f}" for k in ("base", "sft")) + \
           f" | **{S['dpo'][key]['value']:.1f}** ({S['dpo'][key]['ci_low']:.1f}–{S['dpo'][key]['ci_high']:.1f}) |"
pat = lambda k: f"| {k} | {bd['call_pattern'][k]['n']:,} | {bd['call_pattern'][k]['exact_match']:.1f}% |"

card = f"""---
base_model: Qwen/Qwen2.5-0.5B-Instruct
library_name: transformers
license: apache-2.0
pipeline_tag: text-generation
language: [en]
tags: [function-calling, tool-use, qlora, dpo, unsloth, trl, small-language-model]
datasets: [Salesforce/xlam-function-calling-60k]
---

# {HF_REPO_ID.split("/")[-1]}

**Qwen2.5-0.5B-Instruct** trained for **function calling** in two stages:
1. **SFT** (QLoRA, Unsloth) on xLAM-60k: learns to write correct tool calls.
2. **DPO** on the SFT model's **own mistakes**: learns to fix its typical argument errors and to answer `[]` when **no tool fits**.

Give it a list of tools and a request; it returns the tool call(s) to make as a **JSON array**, or `[]` if none of the tools can help.
Small enough to run on a laptop CPU or a 6 GB GPU.

Code, data pipeline, evaluation and serving stack (vLLM + LangGraph gateway with guardrails + Langfuse): {GITHUB_URL}

## Results
Same frozen test sets, same prompt, greedy decoding, same scoring code for all three models.

| Metric (%) | Base | SFT | **DPO (this model)** (95% CI) |
|---|---|---|---|
{metric_row("Exact match, {:,} questions ↑".format(m["n"]), "exact_match")}
{metric_row("Function match ↑", "function_match")}
{metric_row("Argument F1 ↑", "argument_f1")}
| **Answers `[]` when no tool fits**, 300 questions ↑ | {ab["base"]["abstain_pct"]:.1f} | {ab["sft"]["abstain_pct"]:.1f} | **{ab["dpo"]["abstain_pct"]:.1f}** ({ab["dpo"]["ci_95"]}) |
| Wrongly answers `[]` when a tool was needed ↓ | {oa["base"]:.1f} | {oa["sft"]:.1f} | {oa["dpo"]:.1f} |
| Plain chat answered as tool JSON ↓ | {chat["base"]:.1f} | {chat["sft"]:.1f} | {chat["dpo"]:.1f} |

- **SFT** lifted exact match {vs_base["a"]}% → {S["sft"]["exact_match"]["value"]:.1f}%, but xLAM has no "no tool fits" examples, so it learned to *always* call a tool (abstention {ab["base"]["abstain_pct"]:.0f}% → {ab["sft"]["abstain_pct"]:.0f}%).
- **DPO** brought abstention back to **{ab["dpo"]["abstain_pct"]:.0f}%** ({ab["dpo"]["removed_pct"]:.0f}% when look-alike tools are present, {ab["dpo"]["swapped_pct"]:.0f}% when the tools are unrelated) **and** raised exact match to {vs_sft["b"]}% ({vs_sft["fixed"]} questions fixed, {vs_sft["broke"]} broken vs SFT, McNemar p = {vs_sft["mcnemar_p"]}).

Example: *"Tell me a joke"* with only a `get_weather` tool → base and SFT call `get_weather(city="New York")`, this model returns `[]`.

| Call pattern | n | Exact match |
|---|---|---|
{pat("single")}
{pat("parallel_same_tool")}
{pat("multi_distinct_tools")}
{pat("mixed")}

## Prompt format (important)
The model was trained on **this exact format**. It is *not* Qwen's native `<tool_call>` format, and it does not read OpenAI-style `tools=` parameters.

**System:**
```
{SYSTEM_PROMPT.strip()}
```
**User:**
```
Available tools:

<tools as a JSON list: name, description, parameters>

User request:

<the request>

Return the required tool call(s) as JSON.
```
**Assistant:** `[{{"name": "get_weather", "arguments": {{"city": "Pune"}}}}]`, or `[]` if no tool fits.

## Usage
### transformers
```python
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

repo = "{HF_REPO_ID}"
tok = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo, dtype="auto", device_map="auto")

SYSTEM_PROMPT = \"\"\"<copy the system prompt above>\"\"\"
tools = [{{"name": "get_weather", "description": "Current weather for a city.",
          "parameters": {{"city": {{"type": "str", "description": "City name"}}}}}}]
user = (f"Available tools:\\n\\n{{json.dumps(tools, indent=2)}}\\n\\n"
        f"User request:\\n\\nWhat's the weather in Pune?\\n\\n"
        f"Return the required tool call(s) as JSON.")

messages = [{{"role": "system", "content": SYSTEM_PROMPT}}, {{"role": "user", "content": user}}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                 return_tensors="pt", return_dict=True).to(model.device)
out = model.generate(**inputs, max_new_tokens=256)          # greedy is the default
calls = json.loads(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print(calls if calls else "no tool fits this request")
```

### vLLM (OpenAI-compatible server)
```bash
vllm serve {HF_REPO_ID} --max-model-len 4096
```
Send the same two messages to `/v1/chat/completions` with **`temperature=0`**, then `json.loads` the reply.

### Ollama (GGUF, Q8_0)
```bash
ollama run Avahan/qwen2.5_0.5b-instruct-fc-Chat:dpo
```
No system prompt is baked in: send the system prompt above yourself. (`:latest` is this DPO model too; `:sft` is the earlier SFT-only version.)

### Decoding
Use **greedy decoding**. `generation_config.json` already sets `do_sample=False`; with OpenAI-compatible
servers pass `temperature=0`. Sampling and repetition penalties hurt JSON accuracy. Cap `max_new_tokens` (~256-512).

## How it behaves
- **With the system prompt above:** answers with a JSON array of tool calls, or `[]` when none of the tools fit. It's a tool-calling specialist, not a chatbot.
- **Without it** (plain chat): answers in normal text ({chat["dpo"]:.0f}% of 50 everyday prompts turned into tool JSON), but chat quality is that of a 0.5B model: don't rely on it for facts.
  A common pattern (used in the GitHub gateway): send the request with the tool prompt first; if the model returns `[]`, answer it as plain chat.
- **It does not execute anything.** It only writes the call; your code runs the tool. Validate the JSON against your tool
  schemas before running it, and ask the user before running tools with side effects.

## Limitations
- **Abstention isn't perfect:** it still calls a tool in ~{100 - ab["dpo"]["abstain_pct"]:.0f}% of no-tool-fits cases, more often when a similar-looking tool exists. Keep a guard (e.g. check that argument values appear in the request).
- Most remaining errors are **wrong argument values** ({m["failure_mix"]["wrong_value"]:.0f}% of questions), often optional/default arguments where the training data itself is inconsistent.
- Rarely (~{m["failure_mix"].get("parse_fail", 0):.1f}%) it loops on long repeated values until `max_new_tokens`; cap the output length.
- **Unseen tools:** almost every tool in the test set also appears in training, so the score measures new *requests* for familiar tools. Accuracy on completely new tool schemas may be lower.
- Single turn only: it picks the call; it doesn't read tool results or write a final natural-language answer. English only.

## Training
- **SFT:** QLoRA (4-bit base) with Unsloth, r=16, alpha=32, all linear layers, 1 epoch on ~48.7k xLAM examples, loss on the answer only. Merged into bf16.
- **DPO:** 1,753 preference pairs built automatically from held-out xLAM prompts (never used in SFT):
  - *mistake pairs*: chosen = correct answer, rejected = the SFT model's **own most common wrong answer** (sampled 4× at temperature 0.8)
  - *abstain pairs*: tools that can't answer the request; chosen = `[]`, rejected = the call the model invented
  - *balance pairs*: same requests with the right tools; chosen = the call, rejected = `[]` (so it doesn't learn "always say `[]`")
- New LoRA (r=16, alpha=32) on the merged SFT model; reference = the same model with the adapter off. TRL DPO, beta 0.1, RPO alpha 1.0 (keeps correct answers likely), lr 2e-5, 2 epochs.
  Held-out preference accuracy 0.49 → 0.78. Merged into bf16.
"""
(MERGED_DIR / "README.md").write_text(card, encoding="utf-8")
print(card[:3500])


## 6. Log in
Paste your **write** token when the box appears. It's saved on your PC, so you only do this once.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 7. Upload
`create_repo` makes the repo (`exist_ok=True`: re-running just updates it), and `upload_folder` sends the whole `merged/` folder, including the new `README.md` and `generation_config.json`. ~1 GB takes a few minutes.

Flip `PRIVATE = False` in step 1 when you're ready to make it public, or change it later on the website (Settings → visibility).


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_REPO_ID, private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=str(MERGED_DIR), repo_id=HF_REPO_ID,
                  commit_message=f"Upload {RUN_NAME} (SFT + DPO, merged LoRAs)")
print(f"✅ Done: https://huggingface.co/{HF_REPO_ID}")

## 8. Check: load it back from the Hub
This downloads the model the way any user would and runs the same test as step 4, **without** passing `do_sample`, so it also proves the greedy default from step 3 made it to the Hub.


In [ ]:
tok_hub = AutoTokenizer.from_pretrained(HF_REPO_ID)
model_hub = AutoModelForCausalLM.from_pretrained(HF_REPO_ID, dtype="auto", device_map="auto")
print("Hub generation config: do_sample =", model_hub.generation_config.do_sample)

inputs = tok_hub.apply_chat_template(build_messages(record), add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(model_hub.device)
out = getattr(model_hub, "generate")(**inputs, max_new_tokens=128)   # no do_sample: uses the repo default
print(tok_hub.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
